In [1]:
from pathlib import Path
import importlib
import src.flow.builder as fb
importlib.reload(fb)
from src.flow.builder import FlowBuilder

import json
import hashlib
import pandas as pd
import numpy as np
import yaml

from src.utils.paths import load_paths
from src.utils.logging import setup_logger
from src.datasets.pcap_reader import iter_packets

In [2]:
paths = load_paths()
paths.ensure_dirs()
logger = setup_logger(level="INFO")

logger.info(f"Repo root: {paths.repo_root}")
logger.info(f"ISCX raw dir: {paths.data_raw / 'iscx'}")
logger.info(f"Processed dir: {paths.data_processed}")

2026-02-28 18:55:13 | INFO | ai-vpn-firewall | Repo root: C:\Users\scoti\PycharmProjects\ai-vpn-firewall
2026-02-28 18:55:13 | INFO | ai-vpn-firewall | ISCX raw dir: C:\Users\scoti\PycharmProjects\ai-vpn-firewall\data\raw\iscx
2026-02-28 18:55:13 | INFO | ai-vpn-firewall | Processed dir: C:\Users\scoti\PycharmProjects\ai-vpn-firewall\data\processed


In [3]:
features_path = paths.configs_dir / "features.yaml"
assert features_path.exists(), f"Missing: {features_path}"

features_cfg = yaml.safe_load(features_path.read_text()) or {}
w = features_cfg.get("window") or {}

N = int(w.get("N", 100))
EPS = float(w.get("eps", 1e-6))
MIN_PACKETS = int(w.get("min_packets", 10))

logger.info(f"Loaded features config: N={N}, eps={EPS}, min_packets={MIN_PACKETS}")

2026-02-28 18:55:13 | INFO | ai-vpn-firewall | Loaded features config: N=100, eps=1e-06, min_packets=3


In [4]:
raw_iscx = paths.data_raw / "iscx"
vpn_dir = raw_iscx / "vpn"
nonvpn_dir = raw_iscx / "nonvpn"

assert vpn_dir.exists(), f"Missing folder: {vpn_dir}"
assert nonvpn_dir.exists(), f"Missing folder: {nonvpn_dir}"

def list_pcaps(d: Path):
    exts = {".pcap", ".pcapng"}
    return sorted([p for p in d.rglob("*") if p.suffix.lower() in exts])

vpn_pcaps = list_pcaps(vpn_dir)
nonvpn_pcaps = list_pcaps(nonvpn_dir)

logger.info(f"VPN pcaps: {len(vpn_pcaps)}")
logger.info(f"NonVPN pcaps: {len(nonvpn_pcaps)}")

vpn_pcaps[:5], nonvpn_pcaps[:5]

2026-02-28 18:55:13 | INFO | ai-vpn-firewall | VPN pcaps: 31
2026-02-28 18:55:13 | INFO | ai-vpn-firewall | NonVPN pcaps: 23


([WindowsPath('C:/Users/scoti/PycharmProjects/ai-vpn-firewall/data/raw/iscx/vpn/vpn_aim_chat1a.pcap'),
  WindowsPath('C:/Users/scoti/PycharmProjects/ai-vpn-firewall/data/raw/iscx/vpn/vpn_aim_chat1b.pcap'),
  WindowsPath('C:/Users/scoti/PycharmProjects/ai-vpn-firewall/data/raw/iscx/vpn/vpn_bittorrent.pcap'),
  WindowsPath('C:/Users/scoti/PycharmProjects/ai-vpn-firewall/data/raw/iscx/vpn/vpn_email2a.pcap'),
  WindowsPath('C:/Users/scoti/PycharmProjects/ai-vpn-firewall/data/raw/iscx/vpn/vpn_email2b.pcap')],
 [WindowsPath('C:/Users/scoti/PycharmProjects/ai-vpn-firewall/data/raw/iscx/nonvpn/aim_chat_3a.pcap'),
  WindowsPath('C:/Users/scoti/PycharmProjects/ai-vpn-firewall/data/raw/iscx/nonvpn/aim_chat_3b.pcap'),
  WindowsPath('C:/Users/scoti/PycharmProjects/ai-vpn-firewall/data/raw/iscx/nonvpn/AIMchat1.pcapng'),
  WindowsPath('C:/Users/scoti/PycharmProjects/ai-vpn-firewall/data/raw/iscx/nonvpn/AIMchat2.pcapng'),
  WindowsPath('C:/Users/scoti/PycharmProjects/ai-vpn-firewall/data/raw/iscx/nonv

In [5]:
def make_non_decreasing(ts, eps: float):
    out = []
    prev = None
    for t in ts:
        t = float(t)
        if prev is None:
            out.append(t)
            prev = t
            continue
        if t < prev:
            t = prev + eps
        out.append(t)
        prev = t
    return out

def normalize_capture_name(name: str) -> str:
    s = str(name).strip().lower().replace("\\", "/").split("/")[-1]
    return s

def derive_app_from_prefixed_filename(fname: str) -> str:
    s = normalize_capture_name(fname)
    s = s.replace(".pcapng", "").replace(".pcap", "")
    s = s.replace("vpn_", "").replace("nonvpn_", "")
    return s.split("_", 1)[0]

In [6]:
test_pcap = nonvpn_pcaps[0]
logger.info(f"Testing one PCAP: {test_pcap}")

builder = FlowBuilder(tcp_timeout=1800.0, udp_timeout=60.0)

n_pkts = 0
for rec in iter_packets(test_pcap):
    builder.add_packet(**rec)
    n_pkts += 1

flows_list = builder.finalize()
logger.info(f"Packets read: {n_pkts}")
logger.info(f"Flows built: {len(flows_list)}")

flows_list[0].keys(), flows_list[0]["connection"]

2026-02-28 18:55:13 | INFO | ai-vpn-firewall | Testing one PCAP: C:\Users\scoti\PycharmProjects\ai-vpn-firewall\data\raw\iscx\nonvpn\aim_chat_3a.pcap
2026-02-28 18:55:14 | INFO | ai-vpn-firewall | Packets read: 1225
2026-02-28 18:55:14 | INFO | ai-vpn-firewall | Flows built: 55


(dict_keys(['connection', 'timestamps', 'sizes', 'directions']),
 ('131.202.240.87', 64716, '131.202.6.26', 13111, 6))

In [7]:
rows = []

def process_pcap(pcap_path: Path, label: int):
    pref = "vpn_" if label == 1 else "nonvpn_"
    base = pcap_path.name
    file_name = pref + base

    builder = FlowBuilder(tcp_timeout=1800.0, udp_timeout=60.0)

    pkt_count = 0
    for rec in iter_packets(pcap_path):
        builder.add_packet(**rec)
        pkt_count += 1

    built = builder.finalize()

    for f in built:
        ts = make_non_decreasing(f["timestamps"], eps=EPS)
        rows.append({
            "connection": f["connection"],
            "timestamps": ts,
            "sizes": f["sizes"],
            "directions": f["directions"],
            "file_names": file_name,
            "label": int(label),
        })

    return pkt_count, len(built)

total_pkts = 0
total_flows = 0

for i, p in enumerate(nonvpn_pcaps):
    pk, fl = process_pcap(p, label=0)
    total_pkts += pk
    total_flows += fl
    if (i + 1) % 10 == 0:
        logger.info(f"NonVPN processed: {i+1}/{len(nonvpn_pcaps)} | pkts={total_pkts} flows={total_flows}")

for i, p in enumerate(vpn_pcaps):
    pk, fl = process_pcap(p, label=1)
    total_pkts += pk
    total_flows += fl
    if (i + 1) % 10 == 0:
        logger.info(f"VPN processed: {i+1}/{len(vpn_pcaps)} | pkts={total_pkts} flows={total_flows}")

logger.info(f"TOTAL pkts={total_pkts} TOTAL flows={total_flows}")

df = pd.DataFrame(rows)
df.head(), df.shape

2026-02-28 18:56:46 | INFO | ai-vpn-firewall | NonVPN processed: 10/23 | pkts=142984 flows=9100
2026-02-28 19:18:58 | INFO | ai-vpn-firewall | NonVPN processed: 20/23 | pkts=2648354 flows=113944
2026-02-28 19:22:57 | INFO | ai-vpn-firewall | VPN processed: 10/31 | pkts=3320126 flows=119302
2026-02-28 19:33:49 | INFO | ai-vpn-firewall | VPN processed: 20/31 | pkts=5655126 flows=133061
2026-02-28 19:39:45 | INFO | ai-vpn-firewall | VPN processed: 30/31 | pkts=7275809 flows=137536
2026-02-28 19:40:20 | INFO | ai-vpn-firewall | TOTAL pkts=7484145 TOTAL flows=137989


(                                        connection  \
 0  (131.202.240.87, 64716, 131.202.6.26, 13111, 6)   
 1  (131.202.240.87, 64716, 131.202.6.26, 13111, 6)   
 2  (131.202.240.87, 137, 131.202.243.255, 137, 17)   
 3  (131.202.240.87, 64717, 131.202.6.26, 13000, 6)   
 4  (131.202.240.87, 64717, 131.202.6.26, 13000, 6)   
 
                                           timestamps  \
 0  [1430325851.587466, 1430325851.588106, 1430325...   
 1             [1430325851.589252, 1430325851.589283]   
 2  [1430325829.347787, 1430325830.097696, 1430325...   
 3  [1430326173.280069, 1430326173.280739, 1430326...   
 4             [1430326173.729931, 1430326173.730012]   
 
                                                sizes  \
 0                           [66, 66, 54, 56, 60, 54]   
 1                                           [60, 60]   
 2                                       [92, 92, 92]   
 3  [66, 66, 54, 560, 1404, 497, 54, 244, 280, 140...   
 4                                     

In [8]:
df = df.reset_index(drop=True)
df["row_id"] = df.index.astype("int64")

expected_cols = {"connection", "timestamps", "sizes", "directions", "file_names", "label", "row_id"}
missing = expected_cols - set(df.columns)
assert not missing, f"Missing columns: {missing}"

def is_listlike(x): return isinstance(x, (list, tuple))

for col in ["timestamps", "sizes", "directions"]:
    bad = df[~df[col].map(is_listlike)]
    assert len(bad) == 0, f"Non-list entries in {col}: {bad.head()}"

lens = pd.DataFrame({
    "t": df["timestamps"].map(len),
    "s": df["sizes"].map(len),
    "d": df["directions"].map(len),
})
mismatch = df[(lens["t"] != lens["s"]) | (lens["t"] != lens["d"])]
assert len(mismatch) == 0, f"Mismatched list lengths:\n{mismatch.head()}"

bad_dir = df[~df["directions"].map(lambda dirs: set(dirs).issubset({0,1}))]
assert len(bad_dir) == 0, f"Invalid direction values:\n{bad_dir.head()}"

def sizes_valid(sz) -> bool:
    if any(x is None for x in sz):
        return False
    return all((isinstance(x, (int, float)) and x >= 0 and x < 65536) for x in sz)

bad_sizes = df[~df["sizes"].map(sizes_valid)]
assert len(bad_sizes) == 0, f"Invalid sizes:\n{bad_sizes[['file_names','sizes']].head()}"

df["file_names"] = df["file_names"].astype(str)
df["capture_name"] = df["file_names"].map(normalize_capture_name)
df["capture_id"] = df["capture_name"]

def conn_to_str(conn) -> str:
    try:
        ip_a, port_a, ip_b, port_b, proto = conn
        return f"{ip_a}:{int(port_a)}-{ip_b}:{int(port_b)}-p{int(proto)}"
    except Exception:
        return str(conn)

df["connection_str"] = df["connection"].map(conn_to_str)
df["flow_key"] = df["connection_str"]
df["flow_id"] = df["capture_id"] + "::" + df["row_id"].astype(str)

df["app"] = df["file_names"].map(derive_app_from_prefixed_filename)

df["packet_count_full"] = df["sizes"].map(len)

# windowing
df["timestamps"] = df["timestamps"].map(lambda xs: xs[:N])
df["sizes"] = df["sizes"].map(lambda xs: xs[:N])
df["directions"] = df["directions"].map(lambda xs: xs[:N])

df["packet_count"] = df["sizes"].map(len)
df["window_complete"] = df["packet_count_full"] >= N
df["min_packets_ok"] = df["packet_count"] >= MIN_PACKETS

logger.info(f"ISCX flows built: shape={df.shape}")
logger.info("Label counts:\n" + str(df["label"].value_counts()))
logger.info(f"min_packets_ok rate: {100*df['min_packets_ok'].mean():.2f}%")

2026-02-28 19:40:51 | INFO | ai-vpn-firewall | ISCX flows built: shape=(137989, 17)
2026-02-28 19:40:51 | INFO | ai-vpn-firewall | Label counts:
label
0    114094
1     23895
Name: count, dtype: int64
2026-02-28 19:40:51 | INFO | ai-vpn-firewall | min_packets_ok rate: 4.93%


In [9]:
# --- SANITY CHECK: Verify Label Semantics ---
# Check if "vpn_" filenames map to label 1 and "nonvpn_" to label 0
print("\n--- SANITY CHECK: Label vs Filename ---")
df["is_vpn_file"] = df["file_names"].str.startswith("vpn_")
crosstab = pd.crosstab(df["is_vpn_file"], df["label"])
print(crosstab)

if crosstab.shape == (2, 2):
    # Ideal case:
    # is_vpn_file | 0 | 1
    # False       | N | 0
    # True        | 0 | M

    nonvpn_ok = crosstab.loc[False, 0] > 0 and crosstab.loc[False, 1] == 0
    vpn_ok = crosstab.loc[True, 1] > 0 and crosstab.loc[True, 0] == 0

    if nonvpn_ok and vpn_ok:
        print("\nSUCCESS: Labels match filename prefixes perfectly.")
    else:
        print("\nWARNING: Mismatch detected between filename prefix and label!")
else:
    print("\nWARNING: Crosstab shape is unexpected (maybe missing a class?). Check manually.")

# Check app distribution by label
print("\n--- App Distribution by Label ---")
print(pd.crosstab(df["app"], df["label"]))


--- SANITY CHECK: Label vs Filename ---
label             0      1
is_vpn_file               
False        114094      0
True              0  23895

SUCCESS: Labels match filename prefixes perfectly.

--- App Distribution by Label ---
label                  0      1
app                            
aim                    0     38
bittorrent             0   1006
email2a                0    341
email2b                0    348
facebook               0   3078
ftps                   0    397
hangouts               0  12156
icq                    0     37
netflix                0    466
nonaim               392      0
nonaimchat1           32      0
nonaimchat2           36      0
nonemail1a          3731      0
nonemail1b          3740      0
nonemail2a           455      0
nonemail2b           416      0
nonfacebook       105142      0
nonfacebookchat1      38      0
nonfacebookchat2      31      0
nonfacebookchat3      81      0
sftp                   0     42
skype                  0   3

In [10]:
flows = df[
    [
        "capture_id",
        "capture_name",
        "row_id",
        "flow_id",
        "flow_key",
        "connection",
        "connection_str",
        "timestamps",
        "sizes",
        "directions",
        "file_names",
        "app",
        "label",
        "packet_count",
        "packet_count_full",
        "window_complete",
        "min_packets_ok",
    ]
].copy()

assert flows["flow_id"].is_unique

flows["timestamps"] = flows["timestamps"].map(lambda xs: [float(x) for x in xs])
flows["sizes"] = flows["sizes"].map(lambda xs: [int(x) for x in xs])
flows["directions"] = flows["directions"].map(lambda xs: [int(x) for x in xs])

# add proto + ports for filtering/diagnostics
flows["proto"] = flows["connection"].map(lambda c: int(c[4]))
flows["port_a"] = flows["connection"].map(lambda c: int(c[1]))
flows["port_b"] = flows["connection"].map(lambda c: int(c[3]))

# Filter UDP noise flows (LLMNR/mDNS/NetBIOS/SSDP)
noise_ports = {5355, 5353, 137, 138, 1900}
is_udp = flows["proto"] == 17
has_noise_port = flows["port_a"].isin(noise_ports) | flows["port_b"].isin(noise_ports)

flows_filtered = flows[~(is_udp & has_noise_port)].copy()
flows_filtered["connection"] = flows_filtered["connection"].astype(str)

print("\nBefore:", len(flows), "flows | min_packets_ok:", flows["min_packets_ok"].mean())
print("After :", len(flows_filtered), "flows | min_packets_ok:", flows_filtered["min_packets_ok"].mean())

print("\npacket_count top 20 (filtered):")
print(flows_filtered["packet_count"].value_counts().head(20))

print("\nmin_packets_ok by label (filtered):")
print(flows_filtered.groupby("label")["min_packets_ok"].mean())


Before: 137989 flows | min_packets_ok: 0.04931552515055548
After : 34556 flows | min_packets_ok: 0.14127792568584327

packet_count top 20 (filtered):
packet_count
2      17886
1      11788
100      670
3        640
7        574
4        479
8        261
11       190
6        140
20       102
31        93
15        87
10        84
5         77
13        70
21        65
16        61
36        61
18        60
29        54
Name: count, dtype: int64

min_packets_ok by label (filtered):
label
0    0.159944
1    0.131191
Name: min_packets_ok, dtype: float64


In [11]:
# Save filtered flows
out_dir = paths.data_processed / "iscx"
out_dir.mkdir(parents=True, exist_ok=True)

flows_path = out_dir / "flows.parquet"

# `connection` is already string now -> parquet-safe
flows_filtered.to_parquet(flows_path, index=False)
logger.info(f"Saved ISCX flows parquet: {flows_path}")

def sha256_file(p: Path) -> str:
    h = hashlib.sha256()
    with open(p, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

manifest = {
    "dataset": "iscx",
    "flows_parquet": str(flows_path),
    "flows_sha256": sha256_file(flows_path),
    "features_yaml": str(features_path),
    "features_yaml_sha256": hashlib.sha256(features_path.read_bytes()).hexdigest(),
    "rows": int(len(flows_filtered)),
    "unique_captures": int(flows_filtered["capture_id"].nunique()),
    "unique_flows": int(flows_filtered["flow_id"].nunique()),
    "label_counts": flows_filtered["label"].value_counts().to_dict(),
    "window": {"N": int(N), "eps": float(EPS), "min_packets": int(MIN_PACKETS)},
    "pct_window_complete": float((flows_filtered["packet_count_full"] >= N).mean() * 100),
    "pct_min_packets_ok": float((flows_filtered["min_packets_ok"]).mean() * 100),
}

manifest_path = out_dir / "flows_manifest.json"
manifest_path.write_text(json.dumps(manifest, indent=2), encoding="utf-8")
logger.info(f"Saved ISCX flows manifest: {manifest_path}")

flows_filtered.head()

2026-02-28 19:40:59 | INFO | ai-vpn-firewall | Saved ISCX flows parquet: C:\Users\scoti\PycharmProjects\ai-vpn-firewall\data\processed\iscx\flows.parquet
2026-02-28 19:40:59 | INFO | ai-vpn-firewall | Saved ISCX flows manifest: C:\Users\scoti\PycharmProjects\ai-vpn-firewall\data\processed\iscx\flows_manifest.json


,capture_id,capture_name,row_id,flow_id,flow_key,connection,connection_str,timestamps,sizes,directions,file_names,app,label,packet_count,packet_count_full,window_complete,min_packets_ok,proto,port_a,port_b
0,nonvpn_aim_chat_3a.pcap,nonvpn_aim_chat_3a.pcap,0,nonvpn_aim_chat_3a.pcap::0,131.202.240.87:64716-131.202.6.26:13111-p6,"('131.202.240.87', 64716, '131.202.6.26', 1311...",131.202.240.87:64716-131.202.6.26:13111-p6,"[1430325851.587466, 1430325851.588106, 1430325...","[66, 66, 54, 56, 60, 54]","[1, 0, 1, 1, 0, 1]",nonvpn_aim_chat_3a.pcap,nonaim,0,6,6,False,True,6,64716,13111
1,nonvpn_aim_chat_3a.pcap,nonvpn_aim_chat_3a.pcap,1,nonvpn_aim_chat_3a.pcap::1,131.202.240.87:64716-131.202.6.26:13111-p6,"('131.202.240.87', 64716, '131.202.6.26', 1311...",131.202.240.87:64716-131.202.6.26:13111-p6,"[1430325851.589252, 1430325851.589283]","[60, 60]","[0, 0]",nonvpn_aim_chat_3a.pcap,nonaim,0,2,2,False,False,6,64716,13111
3,nonvpn_aim_chat_3a.pcap,nonvpn_aim_chat_3a.pcap,3,nonvpn_aim_chat_3a.pcap::3,131.202.240.87:64717-131.202.6.26:13000-p6,"('131.202.240.87', 64717, '131.202.6.26', 1300...",131.202.240.87:64717-131.202.6.26:13000-p6,"[1430326173.280069, 1430326173.280739, 1430326...","[66, 66, 54, 560, 1404, 497, 54, 244, 280, 140...","[1, 0, 1, 1, 0, 0, 1, 1, 0, 1, 1, 1, 0, 0, 0, ...",nonvpn_aim_chat_3a.pcap,nonaim,0,27,27,False,True,6,64717,13000
4,nonvpn_aim_chat_3a.pcap,nonvpn_aim_chat_3a.pcap,4,nonvpn_aim_chat_3a.pcap::4,131.202.240.87:64717-131.202.6.26:13000-p6,"('131.202.240.87', 64717, '131.202.6.26', 1300...",131.202.240.87:64717-131.202.6.26:13000-p6,"[1430326173.729931, 1430326173.730012]","[54, 54]","[1, 1]",nonvpn_aim_chat_3a.pcap,nonaim,0,2,2,False,False,6,64717,13000
5,nonvpn_aim_chat_3a.pcap,nonvpn_aim_chat_3a.pcap,5,nonvpn_aim_chat_3a.pcap::5,131.202.240.87:17208-77.72.169.130:11113-p17,"('131.202.240.87', 17208, '77.72.169.130', 111...",131.202.240.87:17208-77.72.169.130:11113-p17,"[1430325935.367054, 1430325935.505162, 1430325...","[150, 60, 109, 60]","[1, 0, 0, 1]",nonvpn_aim_chat_3a.pcap,nonaim,0,4,4,False,True,17,17208,11113


In [12]:
from src.features.extract import load_feature_config, extract_features_from_flows
from src.pipeline.feature_pipeline import FeaturePipeline
from src.pipeline.artifacts import default_feature_artifacts
from src.splits.io import load_splits

features_yaml = paths.configs_dir / "features.yaml"
cfg = load_feature_config(features_yaml)

flows_loaded = pd.read_parquet(flows_path)
logger.info(f"Loaded ISCX flows (filtered) from parquet: {flows_loaded.shape}")

logger.info("Extracting ISCX features...")
features_raw = extract_features_from_flows(flows=flows_loaded, cfg=cfg)
logger.info(f"Raw ISCX features: {features_raw.shape}")

art = default_feature_artifacts(paths.artifacts_features)

pipe = None
if hasattr(FeaturePipeline, "load"):
    pipe = FeaturePipeline.load(art)
elif hasattr(FeaturePipeline, "load_from_artifacts"):
    pipe = FeaturePipeline.load_from_artifacts(art)
else:
    pipe = FeaturePipeline()
    if hasattr(pipe, "load"):
        pipe = pipe.load(art)
    else:
        raise RuntimeError("FeaturePipeline has no load method. Show me feature_pipeline.py and I’ll adapt this cell.")

features_scaled = pipe.transform(features_raw)

# --- Add q_min_packets_ok (trainable filter) ---
# MIN_PACKETS is already loaded from features.yaml above
if "q_packet_count" not in features_scaled.columns:
    raise KeyError(
        "Missing column 'q_packet_count' in features_scaled. "
        "Check your feature extractor output / naming."
    )

features_scaled["q_min_packets_ok"] = (
    features_scaled["q_packet_count"].astype(int) >= int(MIN_PACKETS)
).astype(float)

train_list = paths.data_splits / "iscx_train_captures.txt"
val_list   = paths.data_splits / "iscx_val_captures.txt"
test_list  = paths.data_splits / "iscx_test_captures.txt"

spl = load_splits(train_list, val_list, test_list)
s_train, s_val, s_test = set(spl["train"]), set(spl["val"]), set(spl["test"])

def _split_of(cid: str) -> str:
    if cid in s_train: return "iscx_train"
    if cid in s_val:   return "iscx_val"
    if cid in s_test:  return "iscx_test"
    return "unknown"

features_scaled["split"] = features_scaled["capture_id"].astype(str).map(_split_of)
assert (features_scaled["split"] != "unknown").all()

features_out = out_dir / "features.parquet"
features_scaled.to_parquet(features_out, index=False)
logger.info(f"Saved ISCX scaled features: {features_out}")

features_scaled.head()

2026-02-28 19:41:00 | INFO | ai-vpn-firewall | Loaded ISCX flows (filtered) from parquet: (34556, 20)
2026-02-28 19:41:00 | INFO | ai-vpn-firewall | Extracting ISCX features...
2026-02-28 19:42:10 | INFO | ai-vpn-firewall | Raw ISCX features: (34556, 58)
2026-02-28 19:42:11 | INFO | ai-vpn-firewall | Saved ISCX scaled features: C:\Users\scoti\PycharmProjects\ai-vpn-firewall\data\processed\iscx\features.parquet


,flow_id,capture_id,label,f_up_pkt_ratio,f_up_byte_ratio,f_iat_burstiness,sz_all_mean,sz_all_std,sz_all_p25,sz_all_median,...,h_iat_all_06,h_iat_all_07,h_iat_all_08,h_iat_all_09,h_iat_all_10,h_iat_all_11,q_packet_count,q_window_complete,q_min_packets_ok,split
0,nonvpn_aim_chat_3a.pcap::0,nonvpn_aim_chat_3a.pcap,0,0.464393,0.408448,-0.834438,-0.591124,-0.634579,-0.256287,-0.329831,...,-0.451661,-0.267703,-0.269856,-0.331068,-0.293743,-0.728244,6.0,0.0,1.0,iscx_train
1,nonvpn_aim_chat_3a.pcap::1,nonvpn_aim_chat_3a.pcap,0,-3.094331,-2.226522,-1.361436,-0.587522,-0.658442,-0.143386,-0.319960,...,-0.451661,-0.267703,-0.269856,-0.331068,-0.293743,-0.728244,2.0,0.0,0.0,iscx_train
2,nonvpn_aim_chat_3a.pcap::3,nonvpn_aim_chat_3a.pcap,0,-0.128728,0.156780,1.492388,2.220785,1.853231,-0.081804,1.506120,...,-0.451661,0.172256,-0.269856,0.015084,-0.293743,-0.728244,27.0,0.0,1.0,iscx_train
3,nonvpn_aim_chat_3a.pcap::4,nonvpn_aim_chat_3a.pcap,0,2.243755,1.851954,-1.361436,-0.619942,-0.658442,-0.266551,-0.349572,...,-0.451661,-0.267703,-0.269856,-0.331068,-0.293743,-0.728244,2.0,0.0,0.0,iscx_train
4,nonvpn_aim_chat_3a.pcap::5,nonvpn_aim_chat_3a.pcap,0,-0.425288,0.033320,-0.381505,-0.399758,-0.482978,-0.143386,-0.199044,...,-0.451661,-0.267703,4.024011,-0.331068,-0.293743,-0.728244,4.0,0.0,1.0,iscx_train


In [13]:
def sha256_file(p: Path) -> str:
    h = hashlib.sha256()
    with p.open("rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

features_manifest = {
    "dataset": "iscx",
    "flows_parquet": str(flows_path.resolve()),
    "flows_sha256": sha256_file(flows_path),
    "features_parquet": str(features_out.resolve()),
    "features_sha256": sha256_file(features_out),
    "features_yaml": str(features_yaml.resolve()),
    "features_yaml_sha256": hashlib.sha256(features_yaml.read_bytes()).hexdigest(),
    "rows": {
        "flows": int(len(flows_loaded)),
        "features": int(len(features_scaled)),
        "trainable_min_packets_ok": int((features_scaled["q_min_packets_ok"] == 1.0).sum()),
    },
    "label_counts": features_scaled["label"].value_counts().to_dict(),
    "schema": {
        "n_columns": int(features_scaled.shape[1]),
        "columns": list(features_scaled.columns),
        "dtypes": {c: str(features_scaled[c].dtype) for c in features_scaled.columns},
    },
}

features_manifest_path = out_dir / "features_manifest.json"
features_manifest_path.write_text(json.dumps(features_manifest, indent=2), encoding="utf-8")
logger.info(f"Saved ISCX features manifest: {features_manifest_path}")

print(json.dumps({
    "features_rows": features_manifest["rows"]["features"],
    "trainable_rows": features_manifest["rows"]["trainable_min_packets_ok"],
    "features_sha256": features_manifest["features_sha256"],
}, indent=2))

2026-02-28 19:42:12 | INFO | ai-vpn-firewall | Saved ISCX features manifest: C:\Users\scoti\PycharmProjects\ai-vpn-firewall\data\processed\iscx\features_manifest.json
{
  "features_rows": 34556,
  "trainable_rows": 4882,
  "features_sha256": "8f8e286694f0dbc4a29b622d643f1af60d3fc4a7794cd5442ce31c26bc1bbb7a"
}


In [14]:
print("Has q_min_packets_ok?", "q_min_packets_ok" in features_scaled.columns)
print("q_* columns:", [c for c in features_scaled.columns if c.startswith("q_")])
print("Columns sample:", list(features_scaled.columns)[:20])

Has q_min_packets_ok? True
q_* columns: ['q_packet_count', 'q_window_complete', 'q_min_packets_ok']
Columns sample: ['flow_id', 'capture_id', 'label', 'f_up_pkt_ratio', 'f_up_byte_ratio', 'f_iat_burstiness', 'sz_all_mean', 'sz_all_std', 'sz_all_p25', 'sz_all_median', 'sz_all_p75', 'sz_up_mean', 'sz_up_std', 'sz_up_p25', 'sz_up_median', 'sz_up_p75', 'sz_down_mean', 'sz_down_std', 'sz_down_p25', 'sz_down_median']


In [15]:
# Final quick stats (use flows_filtered or flows_loaded, not `flows` after dropping connection)
print(flows_filtered["packet_count"].value_counts().head(20))
print("min_packets_ok:", flows_filtered["min_packets_ok"].mean())
print(flows_filtered.groupby("label")["min_packets_ok"].mean())

print("flows:", len(flows_filtered))
print("total packets in flows:", flows_filtered["packet_count_full"].sum())

packet_count
2      17886
1      11788
100      670
3        640
7        574
4        479
8        261
11       190
6        140
20       102
31        93
15        87
10        84
5         77
13        70
21        65
16        61
36        61
18        60
29        54
Name: count, dtype: int64
min_packets_ok: 0.14127792568584327
label
0    0.159944
1    0.131191
Name: min_packets_ok, dtype: float64
flows: 34556
total packets in flows: 7174794


## *Split on ISCX*

In [16]:
from src.utils.paths import load_paths
from src.splits.make_split_iscx import make_iscx_capture_split, write_capture_lists

paths = load_paths()
flows_parquet = paths.data_processed / "iscx" / "flows.parquet"

splits = make_iscx_capture_split(flows_parquet, seed=42)

write_capture_lists(splits, paths.data_splits, prefix="iscx")
print({k: len(v) for k,v in splits.items()})

{'train': 38, 'val': 8, 'test': 8}


In [17]:
import pandas as pd
from src.utils.paths import load_paths

paths = load_paths()

iscx_features_path = paths.data_processed / "iscx" / "features.parquet"
df_iscx = pd.read_parquet(iscx_features_path)

def read_list(p):
    return set(
        line.strip()
        for line in p.read_text(encoding="utf-8").splitlines()
        if line.strip()
    )

train_caps = read_list(paths.data_splits / "iscx_train_captures.txt")
val_caps   = read_list(paths.data_splits / "iscx_val_captures.txt")
test_caps  = read_list(paths.data_splits / "iscx_test_captures.txt")

def assign_split(cid: str) -> str:
    if cid in train_caps: return "train"
    if cid in val_caps:   return "val"
    if cid in test_caps:  return "test"
    return "unassigned"

df_iscx["split"] = df_iscx["capture_id"].astype(str).apply(assign_split)

assert (df_iscx["split"] != "unassigned").all(), "Some ISCX captures not in any split list"

df_iscx.loc[df_iscx["split"] == "val",  "split"] = "iscx_val"
df_iscx.loc[df_iscx["split"] == "test", "split"] = "iscx_test"

print(df_iscx["split"].value_counts())

df_iscx.to_parquet(iscx_features_path, index=False)
print("Saved:", iscx_features_path)

split
train        24278
iscx_val      8007
iscx_test     2271
Name: count, dtype: int64
Saved: C:\Users\scoti\PycharmProjects\ai-vpn-firewall\data\processed\iscx\features.parquet
